# Representação Semântica dos Arquivos AQG

O [notebook 1](aqg_context_analysis.ipynb) mediu o tamanho das representações do AQG e motivou uma camada intermediária de domínio. Este notebook constrói uma primeira versão dessa camada, diretamente dos arquivos originais.

O foco é responder a três perguntas:

1. Como organizar terreno, famílias, objetos posicionados e solução mantendo seus atributos juntos?
2. Como rastrear registros e relações até uma versão específica do AQG?
3. A construção preserva os dados e explicita os vínculos que consegue resolver nos arquivos disponíveis?

A memória aqui é uma representação do projeto em Python. O experimento não chama LLMs, não cria ferramentas para agentes e não implementa memória de conversa. Seu resultado é uma estrutura inspecionável, que poderá alimentar as camadas C2 e C3 posteriormente.

## Representação Proposta

```text
AQG original (C0)
    |
    +-> versão da fonte: hash do arquivo + versão do exportador
    |
    +-> memória semântica (C1)
        ├── terreno: parâmetros, dados processados e elementos
        ├── famílias: tipo, propriedades, partes e blocos externos
        ├── solução: objetos posicionados, indicadores, scores e anotações
        ├── programa arquitetônico
        └── relações: instância -> família, com estado e regra explícitos
```

O núcleo da representação é tipado por domínio. Atributos heterogêneos permanecem agrupados em **registros com origem**, preservando nomes, valores e estruturas compostas. Isso evita converter cada escalar em um fato isolado ou descartar campos que ainda não interpretamos. A tipagem interna de cada indicador poderá ser acrescentada conforme sua semântica for validada.

| Decisão | Comportamento inicial |
| --- | --- |
| Identidade | Hash completo do arquivo + caminho no JSON. Identifica uma ocorrência dentro da versão; não identifica a mesma torre entre versões. |
| Procedência | Cada registro tem sua origem. O caminho de um atributo é obtido a partir da origem do registro. |
| Indicadores | Uma entrada por ocorrência, inclusive indicadores compostos e repetidos. A linha `TOTAL` é preservada como dado declarado. |
| Ausência | Campo ausente, `null`, zero e lista vazia são estados distintos. |
| Unidades | Campos originais de unidade são preservados. Não há inferência nem normalização de unidades nesta versão. |
| Geometria | `geometry`, `faces` e linhas de anotações ficam por referência. Posições e rotações permanecem nos registros. |
| Outros blocos volumosos | `road_network` e grupos de análise do terreno ficam por referência, sem interpretação. |
| Relações | Casamento de família por prefixo de segmento nos dois sentidos. Zero candidatos é ausência; vários candidatos são ambiguidade. |

## 1. Setup e Acervo

In [1]:
import hashlib
import json
import sys
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 90)


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src/cad/model.py").exists():
            return candidate
    raise RuntimeError("Abra o notebook dentro do repositório do projeto.")


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data/aqg"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.cad.model import _matches  # noqa: E402

PATHS = sorted(DATA_DIR.rglob("*.json"))
if not PATHS:
    raise FileNotFoundError(f"Nenhum AQG encontrado em {DATA_DIR}.")

SOURCES = {}
for path in PATHS:
    raw = path.read_bytes()
    SOURCES[str(path.relative_to(DATA_DIR))] = {
        "snapshot": hashlib.sha256(raw).hexdigest(),
        "project": json.loads(raw),
    }

print(f"{len(SOURCES)} AQGs disponíveis para construção da memória.")

24 AQGs disponíveis para construção da memória.


## 2. Estruturas da Memória

Esta é uma primeira versão da camada C1 proposta no notebook 1. A C1 é construída em duas etapas, e este notebook implementa só a primeira:

1. **Registros com procedência (implementada aqui).** O AQG é reorganizado pelo papel de cada parte no projeto (terreno, famílias, objetos posicionados, solução e programa), sem perder dados. Cada registro guarda os atributos como vieram e a origem exata (hash do arquivo + JSON Pointer); a geometria volumosa fica por referência.
2. **Interpretação de domínio (próxima etapa).** Sobre esses registros entram os fatos que dependem de regra: unidades confirmadas, valores derivados com a regra registrada, vínculos como descritivo -> torres, nomes de atributo normalizados e uma identidade estável entre versões.


`Record` mantém um conjunto de atributos e sua origem. `Site`, `Family`, `Placement` e `Solution` organizam esses registros pelo papel que desempenham no projeto. Os índices por ID facilitam referências; listas de indicadores continuam sendo listas.

`SourceRef` também substitui os blocos mantidos fora da memória. O objeto `root` conserva a estrutura restante do AQG, incluindo metadados e campos ainda não interpretados. Assim, a projeção de domínio pode crescer sem perder o acesso aos dados originais.

In [2]:
@dataclass(frozen=True)
class SourceRef:
    snapshot: str
    pointer: str

    @property
    def id(self) -> str:
        return f"{self.snapshot}#{self.pointer}"


@dataclass
class Record:
    source: SourceRef
    data: dict[str, Any]

    @property
    def id(self) -> str:
        return self.source.id


@dataclass
class Family:
    record: Record
    class_path: str | None
    kind: str | None


@dataclass(frozen=True)
class FamilyLink:
    status: Literal["resolved", "missing", "ambiguous"]
    candidates: tuple[str, ...]
    rule: str = "prefixo por segmento nos dois sentidos"


@dataclass
class Placement:
    record: Record
    kind: str | None
    family: FamilyLink | None  # Só PlacedFamily estabelece este vínculo.


@dataclass
class Site:
    record: Record
    inputs: list[Record]
    processed_info: list[Record]
    elements: list[Record]


@dataclass
class Solution:
    record: Record
    placements: dict[str, Placement]
    outputs: list[Record]
    annotations: list[Record]


@dataclass
class SemanticMemory:
    filename: str
    snapshot: str
    exporter_version: str | None
    root: Record
    site: Site
    families: dict[str, Family]
    solution: Solution
    program: list[Record]

## 3. Construção a Partir do AQG

A construção preserva a ordem das listas e as coordenadas originais, incluindo Z. Os caminhos usam a notação JSON Pointer, com escape de `~` e `/`. Nenhum valor de indicador é recalculado.

As relações de família são derivadas explicitamente. Uma instância continua na memória mesmo quando seu vínculo não pode ser resolvido. Descritivos de torre permanecem nos indicadores; sua associação a instâncias não é inferida nesta versão.

In [3]:
MISSING = object()
REFERENCE_PATHS = {
    "/solution/road_network",
    "/site/analysis/inclination_groups",
    "/site/analysis/region_groups",
}


def child_path(parent: str, key: str | int) -> str:
    escaped = str(key).replace("~", "~0").replace("/", "~1")
    return f"{parent}/{escaped}"


def lookup(tree: Any, pointer: str) -> Any:
    value = tree
    for part in pointer.split("/")[1:] if pointer else []:
        key = part.replace("~1", "/").replace("~0", "~")
        if isinstance(value, dict):
            value = value.get(key, MISSING)
        elif isinstance(value, list) and key.isdigit() and int(key) < len(value):
            value = value[int(key)]
        else:
            return MISSING
    return value


def project_data(value: Any, snapshot: str, pointer: str = "") -> Any:
    geometry = pointer.endswith(("/geometry", "/faces"))
    annotation_line = pointer.startswith("/solution/annotations/") and pointer.endswith(
        "/line"
    )
    if isinstance(value, dict | list) and (
        geometry or annotation_line or pointer in REFERENCE_PATHS
    ):
        return SourceRef(snapshot, pointer)
    if isinstance(value, dict):
        return {
            key: project_data(item, snapshot, child_path(pointer, key))
            for key, item in value.items()
        }
    if isinstance(value, list):
        return [
            project_data(item, snapshot, child_path(pointer, i))
            for i, item in enumerate(value)
        ]
    return value


def build_memory(project: dict, filename: str, snapshot: str) -> SemanticMemory:
    projected = project_data(project, snapshot)

    def record(pointer: str) -> Record:
        value = lookup(projected, pointer)
        if not isinstance(value, dict):
            raise ValueError(f"{filename}: esperado objeto em {pointer!r}")
        return Record(SourceRef(snapshot, pointer), value)

    def records(pointer: str) -> list[Record]:
        value = lookup(projected, pointer)
        if value is MISSING or value is None:
            return []  # O estado original permanece no registro pai.
        if not isinstance(value, list):
            raise ValueError(f"{filename}: esperada lista em {pointer}")
        return [record(child_path(pointer, i)) for i in range(len(value))]

    families = {}
    for item in records("/families"):
        properties = item.data.get("properties") or {}
        families[item.id] = Family(
            item, item.data.get("class_path"), properties.get("type")
        )

    placements = {}
    for item in records("/solution/placed_objects"):
        kind = item.data.get("type")
        link = None
        if kind == "PlacedFamily":
            hid = item.data.get("hid")
            candidates = tuple(
                fid
                for fid, family in families.items()
                if isinstance(hid, str)
                and hid
                and isinstance(family.class_path, str)
                and family.class_path
                and (
                    _matches(hid, family.class_path) or _matches(family.class_path, hid)
                )
            )
            status = (
                ("resolved" if len(candidates) == 1 else "ambiguous")
                if candidates
                else "missing"
            )
            link = FamilyLink(status, candidates)
        placements[item.id] = Placement(item, kind, link)

    return SemanticMemory(
        filename=filename,
        snapshot=snapshot,
        exporter_version=project.get("version"),
        root=record(""),
        site=Site(
            record("/site"),
            records("/site/inputs"),
            records("/site/processed_info"),
            records("/site/elements"),
        ),
        families=families,
        solution=Solution(
            record("/solution"),
            placements,
            records("/solution/outputs"),
            records("/solution/annotations"),
        ),
        program=records("/archprog"),
    )


MEMORIES = {
    name: build_memory(source["project"], name, source["snapshot"])
    for name, source in SOURCES.items()
}
print(f"{len(MEMORIES)} representações construídas diretamente dos AQGs.")

24 representações construídas diretamente dos AQGs.


## 4. Inspeção de um Projeto

Selecionamos um projeto com famílias de torre e indicadores compostos apenas para apresentar a estrutura. A construção e a verificação de integridade usam o acervo inteiro.

In [4]:
EXAMPLE = next(
    name
    for name, memory in MEMORIES.items()
    if any(f.kind in {"Tower", "DynamicTower3d"} for f in memory.families.values())
    and any(o.data.get("type") == "DescritivoTorre" for o in memory.solution.outputs)
)
example = MEMORIES[EXAMPLE]
print("Arquivo:", example.filename)
print("Snapshot:", example.snapshot)
print("Versão do exportador:", example.exporter_version)

display(
    pd.DataFrame(
        [
            {"bloco": "parâmetros do terreno", "registros": len(example.site.inputs)},
            {
                "bloco": "dados processados",
                "registros": len(example.site.processed_info),
            },
            {"bloco": "elementos do terreno", "registros": len(example.site.elements)},
            {"bloco": "famílias", "registros": len(example.families)},
            {
                "bloco": "objetos posicionados",
                "registros": len(example.solution.placements),
            },
            {"bloco": "indicadores", "registros": len(example.solution.outputs)},
            {"bloco": "programa", "registros": len(example.program)},
        ]
    )
)

display(
    pd.DataFrame(
        [
            {
                "origem": f.record.source.pointer,
                "tipo": f.kind,
                "class_path": f.class_path,
                "propriedades": list((f.record.data.get("properties") or {}).keys()),
            }
            for f in example.families.values()
        ]
    )
)

tower = next(
    p
    for p in example.solution.placements.values()
    if p.family is not None
    and p.family.status == "resolved"
    and example.families[p.family.candidates[0]].kind in {"Tower", "DynamicTower3d"}
)
display(
    pd.DataFrame(
        [
            {
                "origem": tower.record.source.pointer,
                "tipo": tower.kind,
                "posição": tower.record.data["position"],
                "vínculo": tower.family.status,
                "família": example.families[
                    tower.family.candidates[0]
                ].record.source.pointer,
                "regra": tower.family.rule,
            }
        ]
    )
)

descriptor = next(
    o for o in example.solution.outputs if o.data.get("type") == "DescritivoTorre"
)
display(
    pd.DataFrame(
        [
            {
                "campo": key,
                "valor": value,
                "origem": child_path(descriptor.source.pointer, key),
            }
            for key, value in descriptor.data.items()
        ],
        dtype=object,
    )
)

Arquivo: avaliadas/01/bom_6ace404e4a7a894b01852f19.json
Snapshot: 12a1c0208a27158fba06157614318043c8ead46b2a5f286b68e956f885229ec2
Versão do exportador: 18.0.0


,bloco,registros
0,parâmetros do terreno,5
1,dados processados,3
2,elementos do terreno,7
3,famílias,2
4,objetos posicionados,12
5,indicadores,45
6,programa,7


,origem,tipo,class_path,propriedades
0,/families/0,Tower,/building/torre/U/24/3i/23f,"[atico_computavel, altura_ultimo_tipo, atico_na_altura_maxima, pav_tipo, terreo_com_un..."
1,/families/1,Portaria,/building/portaria,"[area_tecnica, type]"


,origem,tipo,posição,vínculo,família,regra
0,/solution/placed_objects/1,PlacedFamily,"{'x': 17.141373, 'angle': 277.1625, 'y': 59.868496, 'z': 1575.4536234580116}",resolved,/families/0,prefixo por segmento nos dois sentidos


,campo,valor,origem
0,n_total_pavimentos,23,/solution/outputs/16/n_total_pavimentos
1,ac_nao_computavel,5066.03,/solution/outputs/16/ac_nao_computavel
2,total_computavel,15782.8,/solution/outputs/16/total_computavel
3,area_privativa,17160,/solution/outputs/16/area_privativa
4,ac_total,5066.03,/solution/outputs/16/ac_total
5,altura,48.4,/solution/outputs/16/altura
6,percentual,25,/solution/outputs/16/percentual
7,n_pavimentos_unidades,22,/solution/outputs/16/n_pavimentos_unidades
8,area_construida,22226.03,/solution/outputs/16/area_construida
9,elevadores,5,/solution/outputs/16/elevadores


### Origem e Estado dos Atributos

A função abaixo serve à inspeção do notebook. Ela apresenta a diferença entre campo ausente, valor `null`, valor presente e conteúdo mantido por referência. Não é uma ferramenta de agente. Um caminho de campo ausente indica onde ele seria procurado, não uma evidência de que exista no arquivo.

In [5]:
def inspect_field(record: Record, *keys: str) -> dict:
    relative = ""
    for key in keys:
        relative = child_path(relative, key)
    value = lookup(record.data, relative)
    state = (
        "missing"
        if value is MISSING
        else "null"
        if value is None
        else "referenced"
        if isinstance(value, SourceRef)
        else "present"
    )
    return {
        "campo": "/".join(keys),
        "estado": state,
        "valor": None if value is MISSING else value,
        "origem": record.source.pointer + relative,
        "snapshot": record.source.snapshot,
    }


display(
    pd.DataFrame(
        [
            inspect_field(tower.record, "position", "z"),
            inspect_field(tower.record, "base_level_id"),
            inspect_field(example.site.record, "analysis"),
            inspect_field(example.solution.record, "road_network"),
        ]
    ).drop(columns="snapshot")
)

# Exemplo controlado: ausência não é null, zero nem lista vazia.
demonstration = Record(
    SourceRef("exemplo-controlado", ""),
    {
        "nulo": None,
        "zero": 0,
        "lista_vazia": [],
    },
)
display(
    pd.DataFrame(
        [
            inspect_field(demonstration, key)
            for key in ("ausente", "nulo", "zero", "lista_vazia")
        ]
    ).drop(columns="snapshot")
)

,campo,estado,valor,origem
0,position/z,present,1575.453623,/solution/placed_objects/1/position/z
1,base_level_id,null,None,/solution/placed_objects/1/base_level_id
2,analysis,present,"{'avg_inclination': 5.259085633174499, 'inclination_groups': SourceRef(snapshot='12a1c...",/site/analysis
3,road_network,null,None,/solution/road_network


,campo,estado,valor,origem
0,ausente,missing,None,/ausente
1,nulo,null,None,/nulo
2,zero,present,0,/zero
3,lista_vazia,present,[],/lista_vazia


## 5. Integridade da Construção no Acervo

A verificação confere três propriedades: registros mantêm sua ocorrência no arquivo; referências apontam para a fonte correta; reconstruir o JSON a partir da memória e dos blocos referenciados reproduz os dados originais. Isso verifica preservação, não correção urbanística nem interpretação de todos os campos.

A comparação preserva inclusive a distinção entre inteiros e floats na serialização. Formatação textual do JSON não faz parte dessa igualdade; os bytes originais são identificados pelo hash. Pendências de vínculo são contabilizadas, não descartadas.

In [6]:
def restore(value: Any, project: dict, snapshot: str) -> Any:
    if isinstance(value, SourceRef):
        assert value.snapshot == snapshot
        original = lookup(project, value.pointer)
        assert original is not MISSING, value.pointer
        return original
    if isinstance(value, dict):
        return {key: restore(item, project, snapshot) for key, item in value.items()}
    if isinstance(value, list):
        return [restore(item, project, snapshot) for item in value]
    return value


def references(value: Any) -> list[SourceRef]:
    if isinstance(value, SourceRef):
        return [value]
    if isinstance(value, dict):
        return [ref for item in value.values() for ref in references(item)]
    if isinstance(value, list):
        return [ref for item in value for ref in references(item)]
    return []


coverage_rows = []
for name, memory in MEMORIES.items():
    project = SOURCES[name]["project"]
    records = [
        memory.root,
        memory.site.record,
        memory.solution.record,
        *memory.site.inputs,
        *memory.site.processed_info,
        *memory.site.elements,
        *(f.record for f in memory.families.values()),
        *(p.record for p in memory.solution.placements.values()),
        *memory.solution.outputs,
        *memory.solution.annotations,
        *memory.program,
    ]
    assert len({r.id for r in records}) == len(records)
    for record in records:
        assert record.source.snapshot == memory.snapshot
        original = lookup(project, record.source.pointer)
        assert isinstance(original, dict)
        assert restore(record.data, project, memory.snapshot) == original
    reconstructed = restore(memory.root.data, project, memory.snapshot)
    assert json.dumps(reconstructed, sort_keys=True) == json.dumps(
        project, sort_keys=True
    )
    assert len(memory.families) == len(project["families"])
    assert len(memory.solution.placements) == len(project["solution"]["placed_objects"])
    assert len(memory.solution.outputs) == len(project["solution"].get("outputs") or [])
    links = Counter()
    for placement in memory.solution.placements.values():
        link = placement.family
        if link is not None:
            links[link.status] += 1
            assert all(fid in memory.families for fid in link.candidates)
            assert (link.status == "resolved") == (len(link.candidates) == 1)
            assert (link.status == "missing") == (len(link.candidates) == 0)
    coverage_rows.append(
        {
            "arquivo": name,
            "famílias": len(memory.families),
            "objetos": len(memory.solution.placements),
            "indicadores": len(memory.solution.outputs),
            "blocos_referenciados": len(references(memory.root.data)),
            "vínculos_resolvidos": links["resolved"],
            "vínculos_ausentes": links["missing"],
            "vínculos_ambíguos": links["ambiguous"],
            "dados_preservados": True,
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)
display(
    Markdown(
        f"**{len(coverage_df)} arquivos construídos e verificados.** "
        f"{coverage_df['vínculos_resolvidos'].sum()} vínculos de família resolvidos, "
        f"{coverage_df['vínculos_ausentes'].sum()} ausentes e "
        f"{coverage_df['vínculos_ambíguos'].sum()} ambíguos. "
        "A reconstrução dos dados originais conferiu em todos os arquivos."
    )
)

,arquivo,famílias,objetos,indicadores,blocos_referenciados,vínculos_resolvidos,vínculos_ausentes,vínculos_ambíguos,dados_preservados
0,avaliadas/01/bom_6ace404e4a7a894b01852f19.json,2,12,45,160,5,0,0,True
1,avaliadas/01/otimo_6ace4108cfe0941cdf9a9189.json,5,20,53,414,13,0,0,True
2,avaliadas/01/ruim_6ace40534a7a894b01852f1d.json,2,10,44,171,4,0,0,True
3,avaliadas/02/bom_6ace3c8be748be1a355e6769.json,4,10,47,276,4,0,0,True
4,avaliadas/02/otimo_6ace4079bf22db620be742d4.json,3,9,46,191,3,0,0,True
5,avaliadas/02/ruim_6ace3c16d8664cf36a13dbb9.json,3,9,46,159,3,0,0,True
6,avaliadas/03/bom_6ace7b93445b3a60fcb6162b.json,4,83,51,143,82,0,0,True
7,avaliadas/03/otimo_6ace84f467fab4d8049d1394.json,5,105,53,168,104,0,0,True
8,avaliadas/03/ruim_6ace7f0d246ca4a63fa193fd.json,4,23,51,63,22,0,0,True
9,avaliadas/04/bom_6ace64863afd82bf6be93e16.json,3,161,55,181,152,0,0,True


**24 arquivos construídos e verificados.** 3181 vínculos de família resolvidos, 0 ausentes e 0 ambíguos. A reconstrução dos dados originais conferiu em todos os arquivos.

## 6. Exportação de um Exemplo

Para inspeção fora do notebook, a memória do projeto de exemplo é gravada em JSON em `experiments/results/`, pasta fora do versionamento: o arquivo deriva dos AQGs, que também não são versionados.

O formato espelha as estruturas da seção 2. O `snapshot` aparece uma vez no topo, e cada `source` é um JSON Pointer no arquivo identificado por ele. Dois marcadores substituem conteúdo:

- `{"$ref": ponteiro}`: bloco mantido por referência (geometria, malhas, rede viária, grupos de análise);
- `{"$records": ponteiro}`: lista que virou registros próprios, exportados na sua seção (por exemplo, `site.inputs`), para não repetir dados dentro do registro pai.

In [7]:
EXPORT_DIR = REPO_ROOT / "experiments/results"
# Listas que viram registros proprios, por ponteiro do registro pai.
CHILD_RECORDS = {
    "": ("site", "families", "solution", "archprog"),
    "/site": ("inputs", "processed_info", "elements"),
    "/solution": ("placed_objects", "outputs", "annotations"),
}


def to_json(value: Any) -> Any:
    """SourceRef vira {"$ref": ponteiro}; o snapshot fica uma vez no topo."""
    if isinstance(value, SourceRef):
        return {"$ref": value.pointer}
    if isinstance(value, dict):
        return {key: to_json(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_json(item) for item in value]
    return value


def record_json(record: Record) -> dict:
    children = CHILD_RECORDS.get(record.source.pointer, ())
    data = {
        key: (
            {"$records": child_path(record.source.pointer, key)}
            if key in children
            else to_json(value)
        )
        for key, value in record.data.items()
    }
    return {"source": record.source.pointer, "data": data}


def family_link_json(memory: SemanticMemory, link: FamilyLink | None) -> dict | None:
    if link is None:
        return None
    return {
        "status": link.status,
        "candidates": [
            memory.families[fid].record.source.pointer for fid in link.candidates
        ],
        "rule": link.rule,
    }


def memory_json(memory: SemanticMemory) -> dict:
    return {
        "filename": memory.filename,
        "snapshot": memory.snapshot,
        "exporter_version": memory.exporter_version,
        "root": record_json(memory.root),
        "site": {
            "record": record_json(memory.site.record),
            "inputs": [record_json(r) for r in memory.site.inputs],
            "processed_info": [record_json(r) for r in memory.site.processed_info],
            "elements": [record_json(r) for r in memory.site.elements],
        },
        "families": {
            f.record.source.pointer: {
                "kind": f.kind,
                "class_path": f.class_path,
                "record": record_json(f.record),
            }
            for f in memory.families.values()
        },
        "solution": {
            "record": record_json(memory.solution.record),
            "placements": {
                p.record.source.pointer: {
                    "kind": p.kind,
                    "family": family_link_json(memory, p.family),
                    "record": record_json(p.record),
                }
                for p in memory.solution.placements.values()
            },
            "outputs": [record_json(r) for r in memory.solution.outputs],
            "annotations": [record_json(r) for r in memory.solution.annotations],
        },
        "program": [record_json(r) for r in memory.program],
    }


EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_path = EXPORT_DIR / f"memory_{Path(example.filename).stem}.json"
export_path.write_text(
    json.dumps(memory_json(example), ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Arquivo:", export_path.relative_to(REPO_ROOT))
print(f"Tamanho: {export_path.stat().st_size / 1024:.0f} KiB")

Arquivo: experiments/results/memory_bom_6ace404e4a7a894b01852f19.json
Tamanho: 79 KiB
